In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
import pickle

# Đọc movies.dat
movies = pd.read_csv("/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/1m/movies.dat", sep="::", engine="python", 
                     names=["movieId", "title", "genres"], encoding="latin1")

# Đọc ratings.dat và đổi tên cột MovieID -> ItemID
ratings_1m = pd.read_csv("/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/1m/ratings.dat", sep="::", engine="python", 
                      names=['userId', 'movieId', 'rating', 'timestamp'])
# Hiển thị thông tin dữ liệu
print(ratings_1m.head())
print(movies.head())




   userId  movieId  rating  timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
   movieId                               title                        genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy


In [ ]:
# Mapping user/movie
user_mapping = {id: idx for idx, id in enumerate(ratings_1m["userId"].unique())}
movie_mapping = {id: idx for idx, id in enumerate(ratings_1m["movieId"].unique())}

ratings_1m["userId"] = ratings_1m["userId"].map(user_mapping)
ratings_1m["movieId"] = ratings_1m["movieId"].map(movie_mapping)

num_users = len(user_mapping)
num_movies = len(movie_mapping)

# Train/Test split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    ratings_1m[["userId", "movieId"]], ratings_1m["rating"], test_size=0.2, random_state=42
)

# Build NCF model
embedding_size = 50

user_input = layers.Input(shape=(1,))
movie_input = layers.Input(shape=(1,))

user_embedding = layers.Embedding(input_dim=num_users, output_dim=embedding_size)(user_input)
movie_embedding = layers.Embedding(input_dim=num_movies, output_dim=embedding_size)(movie_input)

user_vec = layers.Flatten()(user_embedding)
movie_vec = layers.Flatten()(movie_embedding)

gmf_vec = layers.Multiply()([user_vec, movie_vec])

mlp_user = layers.Dense(128, activation='relu')(user_vec)
mlp_movie = layers.Dense(128, activation='relu')(movie_vec)
mlp_concat = layers.concatenate([mlp_user, mlp_movie])
mlp_output = layers.Dense(64, activation='relu')(mlp_concat)

final_concat = layers.concatenate([gmf_vec, mlp_output])
# Scale output về [1, 5]
output = layers.Dense(1, activation='sigmoid')(final_concat)
output = layers.Lambda(lambda x: x * 4 + 1)(output)  # sigmoid output ∈ (0, 1) → scale về (1, 5)


model = models.Model(inputs=[user_input, movie_input], outputs=output)
model.compile(optimizer='adam', loss='mean_squared_error')

# Train
model.fit([X_train["userId"], X_train["movieId"]], y_train, epochs=15, batch_size=256, verbose=1)

# Lưu embedding
user_embeddings = model.get_layer(index=2).get_weights()[0]
movie_embeddings = model.get_layer(index=3).get_weights()[0]

Epoch 1/15


/Users/chi.nguyenth/miniconda3/envs/myenv/lib/python3.11/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor_26', 'keras_tensor_27']. Received: the structure of inputs=('*', '*')
  warnings.warn(


3126/3126 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - loss: 0.9267
Epoch 2/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - loss: 0.7913
Epoch 3/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.7548
Epoch 4/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.7303
Epoch 5/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - loss: 0.6925
Epoch 6/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.6285
Epoch 7/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 0.5454
Epoch 8/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 0.4734
Epoch 9/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - loss: 0.4225
Epoch 10/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.3861
Epoch 11/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.3618
Epoch 12/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.3430
Epoch 13/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.3295
Epoch 14/15
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - loss: 0.3182
Epoch 15/15
3126/3126 ━━━━

In [32]:
# Lưu model dưới dạng file .h5
model.save("ncf_model.h5")


# Lưu user_mapping
with open("user_mapping.pkl", "wb") as f:
    pickle.dump(user_mapping, f)

# Lưu movie_mapping
with open("movie_mapping.pkl", "wb") as f:
    pickle.dump(movie_mapping, f)



Valid test

In [23]:
# Dự đoán
y_pred = model.predict([X_test["userId"], X_test["movieId"]])
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.4f}")

6252/6252 ━━━━━━━━━━━━━━━━━━━━ 4s 538us/step
Test RMSE: 1.0036


In [21]:
# Chọn userId bất kỳ (ID sau mapping)
test_user = 10

# Lấy các movie đã xem
watched_movie_ids = ratings_1m[ratings_1m["userId"] == test_user]["movieId"].tolist()

# Lấy danh sách movie chưa xem
all_movie_ids = np.arange(num_movies)
unwatched_movie_ids = np.setdiff1d(all_movie_ids, watched_movie_ids)

# Tạo input để dự đoán
user_input = np.full(len(unwatched_movie_ids), test_user).reshape(-1, 1)
movie_input = unwatched_movie_ids.reshape(-1, 1)

# Dự đoán điểm
predicted_ratings = model.predict([user_input, movie_input], verbose=0)

# Lấy top 10 phim
top_indices = predicted_ratings.flatten().argsort()[-10:][::-1]
top_movie_ids = movie_input[top_indices].flatten()

# Map ngược để lấy thông tin tên phim
reverse_movie_mapping = {v: k for k, v in movie_mapping.items()}
top_movie_ids_original = [reverse_movie_mapping[mid] for mid in top_movie_ids]

# Hiển thị tên phim
recommended_movies = movies[movies["movieId"].isin(top_movie_ids_original)]
print("Top 10 recommended movies:")
print(recommended_movies[["title", "genres"]])


Top 10 recommended movies:
                                                  title  \
559                                     Germinal (1993)   
1556                           Conspiracy Theory (1997)   
1859                                    Cimarron (1931)   
2271                              Meet Joe Black (1998)   
2414  Day of the Beast, The (El Día de la bestia) (1...   
2645                                   Wood, The (1999)   
2736                            Mickey Blue Eyes (1999)   
2781                               Public Access (1993)   
2848                                   Body Heat (1981)   
3090                               Fantasia 2000 (1999)   

                               genres  
559                             Drama  
1556  Action|Mystery|Romance|Thriller  
1859                          Western  
2271                          Romance  
2414           Comedy|Horror|Thriller  
2645                            Drama  
2736                   Comedy|Romance  
278

In [28]:
def recommend_movies_for_user(user_id, model, movie_mapping, movies_df, top_n=10):
    """
    Gợi ý top N phim cho một user dựa trên mô hình đã huấn luyện.
    
    Parameters:
        user_id (int): ID của user đã được mapping.
        model (keras.Model): Mô hình NCF đã được train.
        movie_mapping (dict): Ánh xạ movieId gốc -> index trong embedding.
        movies_df (DataFrame): DataFrame chứa thông tin phim gốc (movieId, title, genres).
        top_n (int): Số lượng phim muốn gợi ý.

    Returns:
        DataFrame chứa top N phim được gợi ý (title + genres).
    """
    # Tổng số phim trong model
    num_movies = len(movie_mapping)

    # Tạo danh sách movie_input và user_input
    movie_input = np.arange(num_movies).reshape(-1, 1)
    user_input = np.full((num_movies, 1), user_id)

    # Dự đoán điểm rating
    predicted_ratings = model.predict([user_input, movie_input], verbose=0)

    # Lấy top N phim có điểm cao nhất
    top_indices = predicted_ratings.flatten().argsort()[-top_n:][::-1]
    top_movie_internal_ids = movie_input[top_indices].flatten()

    # Map ngược để lấy movieId gốc
    reverse_movie_mapping = {v: k for k, v in movie_mapping.items()}
    top_movie_ids_original = [reverse_movie_mapping[mid] for mid in top_movie_internal_ids]

    # Trả về thông tin phim
    return movies_df[movies_df["movieId"].isin(top_movie_ids_original)][["title", "genres"]]


In [29]:
test_user_id = 10  # user đã được mapping trong tập 1M
recommended_df = recommend_movies_for_user(test_user_id, model, movie_mapping, movies, top_n=10)
print(recommended_df)


                                   title                          genres
49            Usual Suspects, The (1995)                  Crime|Thriller
315     Shawshank Redemption, The (1994)                           Drama
643   Cold Fever (Á köldum klaka) (1994)                    Comedy|Drama
1232                    Chinatown (1974)      Film-Noir|Mystery|Thriller
1241  Evil Dead II (Dead By Dawn) (1987)  Action|Adventure|Comedy|Horror
1424          Waiting for Guffman (1996)                          Comedy
1683            Big Lebowski, The (1998)   Comedy|Crime|Mystery|Thriller
1791         Character (Karakter) (1997)                           Drama
2502                  Matrix, The (1999)          Action|Sci-Fi|Thriller
2789              American Beauty (1999)                    Comedy|Drama


test = 100k

In [26]:
# Đọc dữ liệu MovieLens 100K
ratings_100k = pd.read_csv('/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/100k/ratings.csv', delimiter=',', names=["userId", "movieId", "rating", "timestamp"], engine='python', skiprows=1)
print(ratings_100k.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [4]:
def find_nearest_embedding_vector(target_vector, embedding_matrix):
    similarities = cosine_similarity([target_vector], embedding_matrix)[0]
    return np.argmax(similarities)

In [5]:
def predict_from_100k(user_id_100k, movie_id_100k,
                      user_map_1m, movie_map_1m,
                      user_emb_1m, movie_emb_1m,
                      model):
    # Tìm user gần nhất trong embedding
    if user_id_100k in user_map_1m:
        user_idx = user_map_1m[user_id_100k]
    else:
        user_vec_100k = np.mean(user_emb_1m, axis=0)  # giả sử trung bình
        user_idx = find_nearest_embedding_vector(user_vec_100k, user_emb_1m)

    # Tìm movie gần nhất trong embedding
    if movie_id_100k in movie_map_1m:
        movie_idx = movie_map_1m[movie_id_100k]
    else:
        movie_vec_100k = np.mean(movie_emb_1m, axis=0)
        movie_idx = find_nearest_embedding_vector(movie_vec_100k, movie_emb_1m)

    # Đưa input về dạng phù hợp cho model
    user_input = np.array([[user_idx]])
    movie_input = np.array([[movie_idx]])

    pred = model.predict([user_input, movie_input], verbose=0)
    return pred[0][0]


In [6]:
user_test = ratings_100k.iloc[0]["userId"]
movie_test = ratings_100k.iloc[0]["movieId"]

predicted = predict_from_100k(
    user_test, movie_test,
    user_mapping, movie_mapping,
    user_embeddings, movie_embeddings,
    model
)

print(f"\n🎯 Dự đoán rating cho user {user_test} và movie {movie_test}: {predicted:.2f}")



🎯 Dự đoán rating cho user 1.0 và movie 1.0: 4.55


/Users/chi.nguyenth/miniconda3/envs/myenv/lib/python3.11/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor', 'keras_tensor_1']. Received: the structure of inputs=('*', '*')
  warnings.warn(


In [7]:
def recommend_top_n_movies_for_user_100k(user_id_100k,
                                         user_map_1m, movie_map_1m,
                                         user_emb_1m, movie_emb_1m,
                                         model, n=10):
    if user_id_100k in user_map_1m:
        user_idx = user_map_1m[user_id_100k]
    else:
        user_idx = find_nearest_embedding_vector(np.mean(user_emb_1m, axis=0), user_emb_1m)

    user_input = np.full(len(movie_emb_1m), user_idx).reshape(-1, 1)
    movie_input = np.arange(len(movie_emb_1m)).reshape(-1, 1)
    preds = model.predict([user_input, movie_input], verbose=0).flatten()

    top_idx = preds.argsort()[-n:][::-1]
    inv_movie_map = {v: k for k, v in movie_map_1m.items()}
    return [(inv_movie_map[idx], preds[idx]) for idx in top_idx]

In [8]:
user_id_test_100k = 1  # user bất kỳ từ tập 100K

top_movies = recommend_top_n_movies_for_user_100k(
    user_id_test_100k,
    user_mapping, movie_mapping,
    user_embeddings, movie_embeddings,
    model, n=5
)

print(f"\n🎬 Gợi ý Top-5 phim cho user {user_id_test_100k}:")
for i, (movie_id, score) in enumerate(top_movies, 1):
    print(f"{i}. MovieID {movie_id} - Predicted rating: {score:.2f}")



🎬 Gợi ý Top-5 phim cho user 1:
1. MovieID 480 - Predicted rating: 5.34
2. MovieID 3247 - Predicted rating: 5.14
3. MovieID 595 - Predicted rating: 4.98
4. MovieID 1028 - Predicted rating: 4.97
5. MovieID 2018 - Predicted rating: 4.96


Chỉ số đánh giá


Thực hiện kiểm thử với MovieLens 100K:

In [24]:
df_100k=ratings_100k
def get_user_movies(user_id, df_100k, threshold=4.0):
    return df_100k[(df_100k["userId"] == user_id) & (df_100k["rating"] >= threshold)]["movieId"].tolist()

def hit_ratio(predicted, actual, top_n=10):
    return sum(1 for i in range(top_n) if predicted[i][0] in actual) / top_n

def ndcg_at_k(predicted, actual, k=10):
    dcg = sum(1 / np.log2(i + 2) for i in range(k) if predicted[i][0] in actual)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(actual), k)))
    return dcg / idcg if idcg != 0 else 0

def precision_at_k(predicted, actual, k=10):
    return sum(1 for i in range(k) if predicted[i][0] in actual) / k

def recall_at_k(predicted, actual, k=10):
    return sum(1 for i in range(k) if predicted[i][0] in actual) / len(actual) if actual else 0

def evaluate_model_on_100k(user_mapping, movie_mapping, user_embeddings, movie_embeddings, model, df_100k, n=10):
    unique_users_100k = df_100k["userId"].unique()
    total_hr = total_ndcg = total_precision = total_recall = 0

    for i, user_id_100k in enumerate(unique_users_100k):
        print(f"Evaluating user {user_id_100k} ({i+1}/{len(unique_users_100k)})")
        actual_movies = get_user_movies(user_id_100k, df_100k)
        if not actual_movies: continue
        top_movies = recommend_top_n_movies_for_user_100k(
            user_id_100k, user_mapping, movie_mapping,
            user_embeddings, movie_embeddings, model, n
        )

        total_hr += hit_ratio(top_movies, actual_movies, top_n=n)
        total_ndcg += ndcg_at_k(top_movies, actual_movies, k=n)
        total_precision += precision_at_k(top_movies, actual_movies, k=n)
        total_recall += recall_at_k(top_movies, actual_movies, k=n)

    num_eval_users = len(unique_users_100k)
    print(f"\n✅ Evaluation Results (Top-{n}):")
    print(f"Average Hit Ratio: {total_hr / num_eval_users:.4f}")
    print(f"Average NDCG: {total_ndcg / num_eval_users:.4f}")
    print(f"Average Precision: {total_precision / num_eval_users:.4f}")
    print(f"Average Recall: {total_recall / num_eval_users:.4f}")

# --- 10. Thực hiện đánh giá ---
evaluate_model_on_100k(user_mapping, movie_mapping, user_embeddings, movie_embeddings, model, df_100k, n=10)

Evaluating user 1 (1/610)
Evaluating user 2 (2/610)
Evaluating user 3 (3/610)
Evaluating user 4 (4/610)
Evaluating user 5 (5/610)
Evaluating user 6 (6/610)
Evaluating user 7 (7/610)
Evaluating user 8 (8/610)
Evaluating user 9 (9/610)
Evaluating user 10 (10/610)
Evaluating user 11 (11/610)
Evaluating user 12 (12/610)
Evaluating user 13 (13/610)
Evaluating user 14 (14/610)
Evaluating user 15 (15/610)
Evaluating user 16 (16/610)
Evaluating user 17 (17/610)
Evaluating user 18 (18/610)
Evaluating user 19 (19/610)
Evaluating user 20 (20/610)
Evaluating user 21 (21/610)
Evaluating user 22 (22/610)
Evaluating user 23 (23/610)
Evaluating user 24 (24/610)
Evaluating user 25 (25/610)
Evaluating user 26 (26/610)
Evaluating user 27 (27/610)
Evaluating user 28 (28/610)
Evaluating user 29 (29/610)
Evaluating user 30 (30/610)
Evaluating user 31 (31/610)
Evaluating user 32 (32/610)
Evaluating user 33 (33/610)
Evaluating user 34 (34/610)
Evaluating user 35 (35/610)
Evaluating user 36 (36/610)
Evaluating